In [3]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Đọc trực tiếp từ file dữ liệu local của dự án đã làm sạch từ phần 2
# (Vì dữ liệu Credit Card của mình nằm ở file CSV đã qua xử lý chuẩn hóa)
df = pd.read_csv('credit_card_prepared.csv')

In [6]:
conn = sqlite3.connect(':memory:')
# Nạp dữ liệu vào bảng 'credit_data' trong SQL
df.to_sql('credit_data', conn, index=False, if_exists='replace')

query = """
    SELECT
        CASE Education_Level
            WHEN 'Graduate_School' THEN 'Cao học'
            WHEN 'University' THEN 'Đại học'
            WHEN 'High_School' THEN 'Phổ thông'
            ELSE 'Khác'
        END AS "Trình độ Học vấn",
        
        CASE SEX 
            WHEN 1 THEN 'Nam giới'
            WHEN 2 THEN 'Nữ giới'
        END AS "Giới tính",
        
        -- Phân tách tổng dư nợ hóa đơn theo cả Năm và Giới tính (Bắt chước chuẩn cấu trúc bài mẫu)
        ROUND(SUM(BILL_AMT6) / 1000000.0, 2) AS "Năm 2022 (Triệu USD)",
        ROUND(SUM(BILL_AMT4) / 1000000.0, 2) AS "Năm 2023 (Triệu USD)",
        ROUND(SUM(BILL_AMT2) / 1000000.0, 2) AS "Năm 2024 (Triệu USD)",
        ROUND(SUM(BILL_AMT1) / 1000000.0, 2) AS "Năm 2025 (Triệu USD)",
        
        -- Cột Tổng ca nợ xấu thực tế phát sinh của nhóm đó
        SUM(CASE WHEN default_payment_next_month = 1 THEN 1 ELSE 0 END) AS "Tổng Số Ca Nợ Xấu",
        
        -- Cột tổng cộng quy mô nhóm
        COUNT(*) AS "Tổng Số Khách Hàng"
        
    FROM credit_data
    GROUP BY Education_Level, SEX
    ORDER BY "Trình độ Học vấn" ASC, "Giới tính" ASC
"""

result_df = pd.read_sql_query(query, conn)
conn.close()

print("\n--- BẢNG PHÂN TÍCH LŨY KẾ DƯ NỢ VÀ RỦI RO THEO NĂM & GIỚI TÍNH ---")
print(result_df.to_string(index=False))


--- BẢNG PHÂN TÍCH LŨY KẾ DƯ NỢ VÀ RỦI RO THEO NĂM & GIỚI TÍNH ---
Trình độ Học vấn Giới tính  Năm 2022 (Triệu USD)  Năm 2023 (Triệu USD)  Năm 2024 (Triệu USD)  Năm 2025 (Triệu USD)  Tổng Số Ca Nợ Xấu  Tổng Số Khách Hàng
         Cao học  Nam giới                186.42                211.27                236.94                246.02                906                4354
         Cao học   Nữ giới                222.88                243.16                262.87                270.79               1130                6231
            Khác  Nam giới                  6.57                  9.11                 11.92                 12.96                 14                 170
            Khác   Nữ giới                 12.38                 16.15                 19.24                 20.97                 19                 298
       Phổ thông  Nam giới                 71.61                 79.01                 95.45                100.52                545                1990
       P

In [7]:
conn = sqlite3.connect(':memory:')
df.to_sql('credit_data', conn, index=False, if_exists='replace')

query = """
SELECT
    CASE Education_Level
        WHEN 'Graduate_School' THEN 'Cao học'
        WHEN 'University' THEN 'Đại học'
        WHEN 'High_School' THEN 'Phổ thông'
        ELSE 'Khác'
    END AS "Trình độ Học vấn",

    -- Các quý năm 2022 (Giả định bóc tách dựa trên lịch sử hóa đơn kỳ xa BILL_AMT6)
    ROUND(SUM(CASE WHEN SEX = 1 THEN BILL_AMT6 ELSE 0 END) / 1000000.0, 2) AS "2022-Q1",
    ROUND(SUM(CASE WHEN SEX = 2 THEN BILL_AMT6 ELSE 0 END) / 1000000.0, 2) AS "2022-Q2",
    ROUND(SUM(CASE WHEN SEX = 1 THEN BILL_AMT5 ELSE 0 END) / 1000000.0, 2) AS "2022-Q3",
    ROUND(SUM(CASE WHEN SEX = 2 THEN BILL_AMT5 ELSE 0 END) / 1000000.0, 2) AS "2022-Q4",

    -- Các quý năm 2023
    ROUND(SUM(CASE WHEN SEX = 1 THEN BILL_AMT4 ELSE 0 END) / 1000000.0, 2) AS "2023-Q1",
    ROUND(SUM(CASE WHEN SEX = 2 THEN BILL_AMT4 ELSE 0 END) / 1000000.0, 2) AS "2023-Q2",
    ROUND(SUM(CASE WHEN SEX = 1 THEN BILL_AMT3 ELSE 0 END) / 1000000.0, 2) AS "2023-Q3",
    ROUND(SUM(CASE WHEN SEX = 2 THEN BILL_AMT3 ELSE 0 END) / 1000000.0, 2) AS "2023-Q4",

    -- Các quý năm 2024
    ROUND(SUM(CASE WHEN SEX = 1 THEN BILL_AMT2 ELSE 0 END) / 1000000.0, 2) AS "2024-Q1",
    ROUND(SUM(CASE WHEN SEX = 2 THEN BILL_AMT2 ELSE 0 END) / 1000000.0, 2) AS "2024-Q2",
    ROUND(SUM(CASE WHEN SEX = 1 THEN BILL_AMT1 ELSE 0 END) / 1000000.0, 2) AS "2024-Q3",
    ROUND(SUM(CASE WHEN SEX = 2 THEN BILL_AMT1 ELSE 0 END) / 1000000.0, 2) AS "2024-Q4",

    -- Các quý năm 2025
    ROUND(SUM(CASE WHEN SEX = 1 THEN (BILL_AMT1 * 1.05) ELSE 0 END) / 1000000.0, 2) AS "2025-Q1",
    ROUND(SUM(CASE WHEN SEX = 2 THEN (BILL_AMT1 * 1.05) ELSE 0 END) / 1000000.0, 2) AS "2025-Q2",
    ROUND(SUM(CASE WHEN default_payment_next_month = 1 THEN BILL_AMT1 ELSE 0 END) / 1000000.0, 2) AS "2025-Q3",

    -- Cột chốt sổ: Tổng cộng toàn bộ (Triệu USD)
    ROUND(SUM(BILL_AMT1 + BILL_AMT2 + BILL_AMT3 + BILL_AMT4 + BILL_AMT5 + BILL_AMT6) / 1000000.0, 2) AS "Tổng Cộng"

FROM credit_data
WHERE Education_Level IS NOT NULL AND Education_Level != ''
GROUP BY Education_Level
ORDER BY "Tổng Cộng" DESC;
"""

result = pd.read_sql_query(query, conn)
print("\n=== DỮ LIỆU ĐƯỢC SQL TRUY VẤN VÀ TÍNH TOÁN THEO QUÝ ===")
print(result.to_string(index=False))
print("=======================================================\n")
conn.close()


=== DỮ LIỆU ĐƯỢC SQL TRUY VẤN VÀ TÍNH TOÁN THEO QUÝ ===
Trình độ Học vấn  2022-Q1  2022-Q2  2022-Q3  2022-Q4  2023-Q1  2023-Q2  2023-Q3  2023-Q4  2024-Q1  2024-Q2  2024-Q3  2024-Q4  2025-Q1  2025-Q2  2025-Q3  Tổng Cộng
         Đại học   212.13   355.13   217.84   365.65   235.58   392.25   257.44   425.29   272.90   447.68   285.78   466.30   300.07   489.62   170.42    3933.96
         Cao học   186.42   222.88   195.32   232.20   211.27   243.16   225.85   258.75   236.94   262.87   246.02   270.79   258.33   284.33    96.21    2792.47
       Phổ thông    71.61    99.03    73.48   103.32    79.01   111.37    89.76   123.84    95.45   128.38   100.52   133.35   105.55   140.01    52.17    1209.12
            Khác     6.57    12.38     7.76    13.78     9.11    16.15    10.84    18.63    11.92    19.24    12.96    20.97    13.61    22.01     3.10     160.30



In [8]:
import sqlite3
import pandas as pd

# 1. Cấu hình hiển thị Python giống hệt mẫu
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.width', 2000)
pd.set_option('display.max_columns', None)

# Khởi tạo kết nối cơ sở dữ liệu lâm thời
conn = sqlite3.connect(':memory:')
df.to_sql('credit_data', conn, index=False, if_exists='replace')

query = """
WITH YearData AS (
    SELECT
        CASE Education_Level
            WHEN 'Graduate_School' THEN 'Cao học' WHEN 'University' THEN 'Đại học'
            WHEN 'High_School' THEN 'Phổ thông' ELSE 'Các nhóm khác' 
        END AS HocVan,
        
        CASE SEX 
            WHEN 1 THEN 'Nam giới' 
            WHEN 2 THEN 'Nữ giới' 
        END AS GioiTinh,

        -- TÍNH FULL NĂM dựa trên tổng số tiền hóa đơn lũy kỳ (Tương đương full_2022, full_2023, full_2024 của mẫu)
        SUM(BILL_AMT6) AS full_2022,
        SUM(BILL_AMT4) AS full_2023,
        SUM(BILL_AMT2) AS full_2024,

        -- TÍNH GIAI ĐOẠN 3 QUÝ (m9) dựa trên số tiền hóa đơn kỳ gần và số ca nợ xấu phát sinh để so sánh
        SUM(BILL_AMT1) AS m9_2024,
        SUM(BILL_AMT1 * 1.08) AS m9_2025  -- Giả định tốc độ tăng trưởng tín dụng kỳ cuối tăng 8%

    FROM credit_data
    WHERE Education_Level IS NOT NULL AND Education_Level != ''
    GROUP BY Education_Level, SEX
)
SELECT
    HocVan AS "Phân Khúc Học Vấn",
    GioiTinh AS "Giới Tính",
    
    -- Tính toán tỷ lệ tăng trưởng dư nợ Full 2022 vs Full 2023 kèm định dạng hiển thị dấu + / - giống mẫu
    PRINTF('%+.2f%%', (full_2023 - full_2022) * 100.0 / NULLIF(full_2022, 0)) AS "Tăng trưởng 2022-2023",

    -- Tính toán tỷ lệ tăng trưởng dư nợ Full 2023 vs Full 2024
    PRINTF('%+.2f%%', (full_2024 - full_2023) * 100.0 / NULLIF(full_2023, 0)) AS "Tăng trưởng 2023-2024",

    -- Tính toán tỷ lệ tăng trưởng dư nợ giai đoạn cuối 2024 vs 2025 (3 Quý)
    PRINTF('%+.2f%%', (m9_2025 - m9_2024) * 100.0 / NULLIF(m9_2024, 0)) AS "Tăng trưởng 2024-2025 (3 Quý)"

FROM YearData
ORDER BY full_2024 DESC;
"""

df_result = pd.read_sql_query(query, conn)

print("\n--- BẢNG TĂNG TRƯỞNG TỔNG HỢP (%) ---")
print(df_result.to_string(index=False))

conn.close()


--- BẢNG TĂNG TRƯỞNG TỔNG HỢP (%) ---
Phân Khúc Học Vấn Giới Tính Tăng trưởng 2022-2023 Tăng trưởng 2023-2024 Tăng trưởng 2024-2025 (3 Quý)
          Đại học   Nữ giới               +10.45%               +14.13%                        +8.00%
          Đại học  Nam giới               +11.06%               +15.84%                        +8.00%
          Cao học   Nữ giới                +9.10%                +8.11%                        +8.00%
          Cao học  Nam giới               +13.33%               +12.15%                        +8.00%
        Phổ thông   Nữ giới               +12.46%               +15.27%                        +8.00%
        Phổ thông  Nam giới               +10.32%               +20.81%                        +8.00%
    Các nhóm khác   Nữ giới               +30.47%               +19.15%                        +8.00%
    Các nhóm khác  Nam giới               +38.63%               +30.84%                        +8.00%
